In [1]:
import pandas as pd
from bs4 import BeautifulSoup
import requests
import numpy as np

In [2]:
headers={'User-Agent':'Mozilla/5.0 (Windows NT 6.3; Win 64 ; x64) Apple WeKit /537.36(KHTML , like Gecko) Chrome/80.0.3987.162 Safari/537.36'}
webpage= requests.get("http://cricbuzz.com/cricket-series/9241/indian-premier-league-2026/matches", headers=headers)


In [8]:
soup= BeautifulSoup(webpage.text, 'lxml')

In [19]:
container= soup.find_all('div', class_="mt-2")
len(container)

0

In [16]:
first= soup.find_all('div', class_="mt-2")
len(first)

0

In [23]:
import requests
from bs4 import BeautifulSoup
import csv

def scrape_cricbuzz_robust(url, output_filename):
    # Added more headers to mimic a real browser even closer
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
        'Accept-Language': 'en-US,en;q=0.9',
        'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8',
    }

    try:
        response = requests.get(url, headers=headers)
        response.raise_for_status()
    except requests.exceptions.RequestException as e:
        print(f"Error fetching the URL: {e}")
        return

    soup = BeautifulSoup(response.content, 'html.parser')
    schedule_data = [['Date', 'Match Info', 'Status/Result', 'Match Link']]

    # MORE ROBUST SELECTOR: Find all links that contain match URLs
    # Cricbuzz match links always include '/live-cricket-scores/'
    match_links = soup.find_all('a', href=lambda href: href and '/live-cricket-scores/' in href)

    for link in match_links:
        # 1. Get the Match Info (e.g., "1st Match • Bengaluru, M.Chinnaswamy Stadium")
        match_info = link.text.strip()
        if not match_info:
            continue

        # 2. Find the parent container to extract the surrounding data (Date and Status)
        # We look up the DOM tree for a common wrapper div (usually a row container)
        parent_row = link.find_parent('div', class_=lambda c: c and 'cb-col' in c)
        
        # 3. Extract the Date
        # The date is usually in a header block preceding the match row
        date = "Unknown Date"
        if parent_row:
            date_header = parent_row.find_previous_sibling('div', class_=lambda c: c and 'cb-lv-grn-strip' in c)
            if date_header:
                date = date_header.text.strip()

        # 4. Extract the Status / Result
        # Cricbuzz places the result in anchor tags with specific text classes next to the teams
        status = "Upcoming / Status Unknown"
        if parent_row:
            status_tag = parent_row.find('a', class_=lambda c: c and ('cb-text-complete' in c or 'cb-text-preview' in c or 'cb-text-live' in c))
            if status_tag:
                status = status_tag.text.strip()

        # 5. Build the full URL
        full_link = "https://www.cricbuzz.com" + link['href']

        schedule_data.append([date, match_info, status, full_link])

    # Write to CSV
    if len(schedule_data) > 1:
        with open(output_filename, mode='w', newline='', encoding='utf-8') as csv_file:
            writer = csv.writer(csv_file)
            writer.writerows(schedule_data)
        print(f"Success! Scraped {len(schedule_data) - 1} matches and saved to '{output_filename}'.")
    else:
        print("Still scraped 0 matches. The page might be rendering dynamically via JavaScript.")
        
        # DEBUGGING: Save the raw HTML to see what BeautifulSoup is actually seeing
        with open("debug_page.html", "w", encoding="utf-8") as f:
            f.write(soup.prettify())
        print("I've saved the raw page to 'debug_page.html'. Open it to see if Cricbuzz blocked the request or changed their HTML.")

if __name__ == "__main__":
    target_url = "https://www.cricbuzz.com/cricket-series/9241/indian-premier-league-2026/matches"
    output_csv = "ipl_2026_schedule_v2.csv"
    scrape_cricbuzz_robust(target_url, output_csv)

PermissionError: [Errno 13] Permission denied: 'ipl_2026_schedule_v2.csv'

In [ ]:
Successfully